# 36 — Incident Postmortem Drafter

Two-stage pipeline: parse a raw incident log into a structured timeline, then draft a blameless postmortem.

In [ ]:
%pip install -q openai pydantic python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()
print('Environment loaded')

In [ ]:
from src.workflow import run

RAW_LOG = """
INCIDENT: INC-2026-042
Title: Database Connection Pool Exhaustion — API Timeouts

2026-06-18T14:30:00Z [INFO] Automated alert: p99 API latency crossed 2000ms threshold (orders-service)
2026-06-18T14:33:00Z [WARNING] orders-service reporting DB connection wait times > 5s; pool utilization 98%
2026-06-18T14:37:00Z [CRITICAL] orders-service returning 503 errors to 60% of checkout requests; DB pool fully exhausted
2026-06-18T14:40:00Z [INFO] On-call engineer paged; investigation started
2026-06-18T14:52:00Z [WARNING] payments-service also degraded — shares same RDS instance read replica
2026-06-18T15:05:00Z [INFO] Root cause identified: batch analytics job deployed at 14:15 consuming 200+ concurrent DB connections
2026-06-18T15:10:00Z [INFO] Analytics job terminated; connections released; orders-service error rate dropping
2026-06-18T15:22:00Z [INFO] API latency returned to baseline; payments-service fully recovered
2026-06-18T16:15:00Z [INFO] Incident resolved; mitigation documented

Affected services: orders-service, payments-service, rds-primary, rds-read-replica
"""

postmortem = run(RAW_LOG)
print(f'Incident: {postmortem.incident_id} — {postmortem.title}')
print(f'Duration: {postmortem.timeline.duration_minutes} minutes')
print(f'Root Cause: {postmortem.root_cause}')

In [ ]:
print('=== Impact ===')
print(postmortem.impact_summary)
print()
print('--- Contributing Factors ---')
for factor in postmortem.contributing_factors:
    print(f'  * {factor}')
print()
print('--- Action Items ---')
for item in postmortem.action_items:
    print(f'  [ ] {item}')
print()
print('--- Detection Improvements ---')
for improvement in postmortem.detection_improvements:
    print(f'  + {improvement}')
print()
print('=== Executive Summary ===')
print(postmortem.executive_summary)

In [ ]:
import json
print(json.dumps(postmortem.model_dump(), indent=2))